# Stage 4 QLoRA Training on Colab
Notebook pentru antrenare reala a adapter-ului LoRA pe `openai/gpt-oss-20b`.


## 0) Runtime
Seteaza runtime: **GPU (T4 sau A100)**.


In [ ]:
!nvidia-smi


In [ ]:
REPO_URL = "https://github.com/alexandraraluca/Hint_generator_Vmchecker.git"  # TODO
PROJECT_DIR = "/content/licenta_vmchecker"

!rm -rf $PROJECT_DIR
!git clone $REPO_URL $PROJECT_DIR
%cd $PROJECT_DIR


In [ ]:
# Versiuni compatibile cu openai/gpt-oss-20b:
#  - transformers 4.55+ (cu suport gpt-oss + harmony chat template)
#  - tokenizers 0.20+    (gpt-oss tokenizer.json folosește features noi)
#  - accelerate 1.x      (transformers 4.55 cere accelerate >=1.0)
# IMPORTANT: NU folosi transformers 5.x (rupe integrarea cu accelerate).

!pip uninstall -y transformers tokenizers accelerate peft trl bitsandbytes
!pip install --no-cache-dir \
    "transformers>=4.55,<5.0" \
    "tokenizers>=0.20,<0.22" \
    "accelerate>=1.0,<2.0" \
    "peft>=0.13,<0.20" \
    "trl>=0.11,<0.25" \
    "bitsandbytes>=0.43" \
    "datasets>=2.20" \
    "sentencepiece" \
    "pyyaml" "orjson"

print("\nDONE installing. Acum: Runtime -> Restart runtime, apoi rerulează celulele de la 'Upload dataset' în jos.")

In [ ]:
# Verificare versiuni (rulează DUPĂ Restart runtime).
# Trebuie să vezi: transformers 4.55+, tokenizers 0.20+, accelerate 1.x, peft 0.13+
import importlib, importlib.metadata as md

for pkg in ["transformers", "tokenizers", "accelerate", "peft", "trl", "bitsandbytes", "datasets"]:
    try:
        v = md.version(pkg)
        print(f"  {pkg:14s} {v}")
    except md.PackageNotFoundError:
        print(f"  {pkg:14s} <not installed>")

import torch
print(f"\n  torch          {torch.__version__} | cuda available: {torch.cuda.is_available()} | device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


In [ ]:
from google.colab import files
import os, shutil

target = f"{PROJECT_DIR}/data/hints"
os.makedirs(target, exist_ok=True)
uploaded = files.upload()  # finetune_train.jsonl, finetune_val.jsonl (+ optional finetune_test.jsonl)

for fn in uploaded.keys():
    shutil.move(fn, os.path.join(target, fn))

print(sorted(os.listdir(target)))


In [ ]:
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONPATH"] = PROJECT_DIR

!python -m src.stage4_finetune.train_qlora --config configs/qlora_colab_t4.yaml --dry-run


In [ ]:
!python -m src.stage4_finetune.train_qlora --config configs/qlora_colab_t4.yaml


In [ ]:
from google.colab import files

ARCHIVE = "/content/gpt_oss_20b_pa_hints_adapter.tgz"
!tar -czf $ARCHIVE -C $PROJECT_DIR/models gpt_oss_20b_pa_hints
!ls -lh $ARCHIVE
files.download(ARCHIVE)
